# stPlus: Spatial Gene Imputation (Pseudo-6K Benchmark)

Uses stPlus to impute missing genes by learning a mapping from spatial cells (6K panel)
to the full transcriptome reference.

**Input**: WTX CosMx h5ad; 6K gene list  
**Output**: `half_ori_half_stplus_imp_log1p.csv` — combined 6K observed + imputed 12K genes


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from stPlus import stPlus

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = "/path/to/cosmx/data"
MODEL_DIR = "/path/to/stplus/model_weights"
OUT_DIR   = "/path/to/cosmx/count_csv"

In [ ]:
# Load WTX CosMx data (log1p normalized)
adata = sc.read_h5ad(f"{DATA_DIR}/qc_wtx_v7_UC_P1_UC_P2_anno.h5ad")
adata.obs_names = adata.obs['fov_cell_id'].tolist()
adata.obs['fov'] = adata.obs['fov'].astype(int)

# Train/test split by FOV
train_cells, test_cells = [], []
for fov, cell in zip(adata.obs['fov'], adata.obs.index):
    if (fov > 9 and fov <= 32) or (fov > 44):
        train_cells.append(cell)
    else:
        test_cells.append(cell)

# DataFrames (stPlus takes pandas DataFrames)
adata_train = pd.DataFrame(adata[train_cells, :].X)
adata_test  = pd.DataFrame(adata[test_cells,  :].X)
adata_train.index   = [str(i) for i in adata[train_cells, :].obs_names]
adata_test.index    = [str(i) for i in adata[test_cells,  :].obs_names]
adata_train.columns = list(adata.var_names)
adata_test.columns  = list(adata.var_names)
print(f"Train: {adata_train.shape}  |  Test: {adata_test.shape}")

In [ ]:
# 6K gene panel (pseudo-6K: restrict test to 6K genes)
genes_6k = pd.read_csv(f"{DATA_DIR}/6k_actual_gene_names.csv")['x'].tolist()
shared_6k = [g for g in genes_6k if g in adata_test.columns]
test_genes = [g for g in adata_test.columns if g not in shared_6k]  # genes to impute

adata_test_partial = adata_test[shared_6k]
print(f"Observed (6K): {len(shared_6k)}  |  To impute: {len(test_genes)}")

In [ ]:
# Impute missing genes with stPlus
stplus_pred = stPlus(adata_test_partial, adata_train, test_genes,
                     save_path_prefix=MODEL_DIR, random_seed=42)
print(stplus_pred.shape)

In [ ]:
# ── Export results ────────────────────────────────────────────────────────────

# Combine: 6K observed counts + stPlus-imputed 12K
combined = pd.concat([adata_test_partial.reset_index(), stplus_pred], axis=1)
combined.index = combined['index'].tolist()
combined = combined.drop('index', axis=1)

combined.to_csv(f"{OUT_DIR}/half_ori_half_stplus_imp_log1p.csv")
print(f"Exported: {combined.shape[0]:,} cells x {combined.shape[1]:,} genes")

---
## Pseudo-1K Section

Same stPlus workflow applied to the pseudo-1K panel (only 1K genes observed; ~17K imputed).
Output used for the per-gene PCC/SCC heatmap in `01_imputation_comparison.ipynb`.


In [ ]:
# 1K gene panel
genes_1k = pd.read_csv(f"{DATA_DIR}/1k_actual_gene_names.csv")['0'].tolist()
shared_1k = [g for g in genes_1k if g in adata_test.columns]
test_genes_1k = [g for g in adata_test.columns if g not in shared_1k]

adata_test_partial_1k = adata_test[shared_1k]
print(f"Observed (1K): {len(shared_1k)}  |  To impute: {len(test_genes_1k)}")

In [ ]:
# stPlus prediction on pseudo-1K
stplus_pred_1k = stPlus(adata_test_partial_1k, adata_train, test_genes_1k,
                        save_path_prefix=MODEL_DIR, random_seed=10)
print(stplus_pred_1k.shape)

In [ ]:
# Export pseudo-1K results
combined_1k = pd.concat([adata_test_partial_1k.reset_index(), stplus_pred_1k], axis=1)
combined_1k.index = combined_1k['index'].tolist()
combined_1k = combined_1k.drop('index', axis=1)

combined_1k.to_csv(f"{OUT_DIR}/1k_ori_17k_stplus_imp_log1p.csv")
print(f"Exported pseudo-1K: {combined_1k.shape[0]:,} cells x {combined_1k.shape[1]:,} genes")